# E2 — YOLO26s + iAFF trên DataSet2

Notebook này là ablation chỉ thêm **iAFF (Iterative Attentional Feature Fusion)** vào các fusion point P3/P4 của Neck.

- Giữ nguyên dataset, `data.yaml`, class order và train/valid split của `Train3.ipynb`.
- Giữ nguyên `yolo26s.pt` và training protocol baseline.
- Không dùng test set trong E2.
- Output riêng: `helmet_yolo26s_iaff_dataset2`; không ghi đè baseline.


In [ ]:
# Cài đúng nhóm dependency mà notebook baseline sử dụng.
!pip install -U ultralytics
!pip install -q gdown
!apt-get update -qq
!apt-get install -y unrar

In [ ]:
# Thiết lập dataset — giữ nguyên dataset và cách chia của Train3.ipynb.
from google.colab import drive
import os
import gdown

drive.mount('/content/drive', force_remount=True)

!mkdir -p /content/DetectedHelmets/DataSet2
!mkdir -p /content/DetectedHelmets/Code
%cd /content/DetectedHelmets

if os.path.exists('dataset.rar'):
    os.remove('dataset.rar')

file_id = '1LUUQLl5F3aczuny4As_PcTtqLIWTmZSJ'
url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, 'dataset.rar', quiet=False, fuzzy=True)
print('Đang giải nén dataset hiện tại...')
!unrar x -o+ dataset.rar /content/DetectedHelmets/DataSet2/
print('Tải xuống và giải nén dataset hoàn tất.')

In [ ]:
# Ưu tiên bản dataset trên Drive nếu có, giống baseline. Không sửa nội dung dataset.
drive_dataset_path = '/content/drive/MyDrive/DataSet2'
local_dataset_path = '/content/DetectedHelmets/DataSet2'

if os.path.exists(drive_dataset_path):
    if os.path.islink(local_dataset_path) or os.path.exists(local_dataset_path):
        !rm -rf {local_dataset_path}
    !ln -s "{drive_dataset_path}" "{local_dataset_path}"
    print(f'Đã liên kết dataset: {local_dataset_path} -> {drive_dataset_path}')
else:
    print(f'Không tìm thấy đường dẫn dataset trên Drive: {drive_dataset_path}')

## 1. Environment, dataset và protocol baseline

In [ ]:
import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import ultralytics
import yaml
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')
assert torch.cuda.is_available(), 'GPU chưa được bật; E2 yêu cầu device=0.'

In [ ]:
CURRENT_DIR = Path(os.getcwd()).resolve()
ROOT_DIR = CURRENT_DIR
while ROOT_DIR.name in ['Train', 'Code'] and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent

DATA_YAML_PATH = ROOT_DIR / 'DataSet2' / 'DataSet2' / 'data.yaml'
MODEL_NAME = 'yolo26s.pt'
OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/runs/train')
EXPERIMENT_NAME = 'helmet_yolo26s_iaff_dataset2'
BASELINE_EXPERIMENT_NAME = 'helmet_yolo26s_dataset2'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'
BASELINE_BEST_MODEL_PATH = OUTPUT_DIR / BASELINE_EXPERIMENT_NAME / 'weights' / 'best.pt'

TRAIN_CONFIG = {
    'epochs': 50,
    'imgsz': 640,
    'batch': 16,
    'device': 0,
    'workers': 4,
    'optimizer': 'auto',
    'seed': 0,
    'deterministic': True,
    'patience': 15,
}

with open(DATA_YAML_PATH, 'r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)

class_names = data_config.get('names', [])
if isinstance(class_names, dict):
    class_names = [class_names[key] for key in sorted(class_names, key=lambda item: int(item))]
CLASS_NAMES = list(class_names)
EXPECTED_CLASS_NAMES = ['motorcycle', 'rider_with_helmet', 'rider_without_helmet']

assert DATA_YAML_PATH.exists(), f'Không tìm thấy data.yaml: {DATA_YAML_PATH}'
assert data_config.get('nc') == 3
assert CLASS_NAMES == EXPECTED_CLASS_NAMES, (CLASS_NAMES, EXPECTED_CLASS_NAMES)
assert EXPERIMENT_NAME != BASELINE_EXPERIMENT_NAME

print(f'Project root   : {ROOT_DIR}')
print(f'data.yaml      : {DATA_YAML_PATH}')
print(f'Pretrained     : {MODEL_NAME}')
print(f'E2 output      : {EXPERIMENT_DIR}')
print(f'Baseline best  : {BASELINE_BEST_MODEL_PATH}')
print(f'nc / names     : {data_config.get("nc")} / {CLASS_NAMES}')
print(f'Train protocol : {TRAIN_CONFIG}')

## 2. Inspect kiến trúc YOLO26s thực tế trong runtime

Không đoán layer index. Cell này in toàn bộ model và probe các `Concat` bằng dummy `640x640`; vị trí P3/P4 được chọn theo spatial stride thực tế.

In [ ]:
base_model = YOLO(MODEL_NAME)
if getattr(base_model, 'task', None) != 'detect':
    raise ValueError(f'YOLO26s phải là detect model, nhận task={base_model.task!r}')

print('--- Kiến trúc YOLO26s đang được cài trong runtime ---')
print(base_model.model)
print('--- model.info() baseline architecture ---')
base_model.info(verbose=True, imgsz=640)

In [ ]:
def probe_concat_points(yolo_model, imgsz=640):
    records = []
    handles = []

    def make_hook(index, module):
        def hook(_module, inputs, output):
            source = inputs[0] if len(inputs) == 1 else inputs
            tensors = list(source) if isinstance(source, (list, tuple)) else [source]
            tensor_shapes = [tuple(t.shape) for t in tensors if torch.is_tensor(t)]
            if not tensor_shapes:
                return
            output_shape = tuple(output.shape) if torch.is_tensor(output) else None
            records.append({
                'index': index,
                'from': getattr(module, 'f', None),
                'input_shapes': tensor_shapes,
                'output_shape': output_shape,
                'height': tensor_shapes[0][-2],
                'width': tensor_shapes[0][-1],
                'stride': imgsz // tensor_shapes[0][-1],
            })
        return hook

    for index, module in enumerate(yolo_model.model.model):
        if module.__class__.__name__ == 'Concat':
            handles.append(module.register_forward_hook(make_hook(index, module)))

    device = next(yolo_model.model.parameters()).device
    was_training = yolo_model.model.training
    yolo_model.model.eval()
    with torch.no_grad():
        yolo_model.model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    if was_training:
        yolo_model.model.train()
    for handle in handles:
        handle.remove()

    return sorted(records, key=lambda record: record['index'])

fusion_probe = probe_concat_points(base_model, imgsz=640)
print('--- Fusion points phát hiện bằng runtime probe ---')
for record in fusion_probe:
    print(record)

# P3/P4 được xác định theo stride 8/16, không theo index hard-code. P5 stride 32 được giữ nguyên.
fusion_specs = [
    {
        'index': record['index'],
        'from': record['from'],
        'input_channels': [shape[1] for shape in record['input_shapes']],
        'spatial': (record['height'], record['width']),
        'stride': record['stride'],
    }
    for record in fusion_probe
    if record['stride'] in (8, 16)
]

assert fusion_specs, 'Không tìm thấy fusion point P3/P4 trong kiến trúc runtime.'
assert any(spec['stride'] == 8 for spec in fusion_specs), 'Thiếu fusion P3/8.'
assert any(spec['stride'] == 16 for spec in fusion_specs), 'Thiếu fusion P4/16.'
print('--- Các fusion point sẽ được thay bằng iAFF ---')
for spec in fusion_specs:
    print(spec)

## 3. Implement iAFF và chèn vào Neck

`iAFF` nhận hai tensor cùng spatial size/channel. `iAFFConcat` là **iAFF-based Concat replacement with 1x1 channel projection**: hai nhánh được project tối thiểu bằng `1x1 Conv` về `c1+c2` channels để output giữ đúng số channel mà các layer sau của YOLO26s đang chờ. P5 không bị thay đổi.

In [ ]:
class iAFF(nn.Module):
    """Iterative Attentional Feature Fusion for two aligned feature maps."""

    def __init__(self, channels, reduction=4):
        super().__init__()
        inter_channels = max(channels // reduction, 8)

        def local_attention():
            return nn.Sequential(
                nn.Conv2d(channels, inter_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(inter_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(inter_channels, channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(channels),
            )

        def global_attention():
            # Canonical iAFF uses BatchNorm2d in both local and global attention paths.
            return nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Conv2d(channels, inter_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(inter_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(inter_channels, channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(channels),
            )

        self.local_att1 = local_attention()
        self.global_att1 = global_attention()
        self.local_att2 = local_attention()
        self.global_att2 = global_attention()

    def forward(self, x, y):
        if x.ndim != 4 or y.ndim != 4:
            raise ValueError(f'iAFF expects BCHW tensors, got {x.shape} and {y.shape}')
        if x.shape != y.shape:
            raise ValueError(f'iAFF inputs must have identical shapes, got {x.shape} and {y.shape}')

        fused = x + y
        weight1 = torch.sigmoid(self.local_att1(fused) + self.global_att1(fused))
        refined = x * weight1 + y * (1.0 - weight1)
        weight2 = torch.sigmoid(self.local_att2(refined) + self.global_att2(refined))
        output = x * weight2 + y * (1.0 - weight2)
        return output


class iAFFConcat(nn.Module):
    """Concat-compatible iAFF fusion preserving the original output channels."""

    def __init__(self, input_channels):
        super().__init__()
        if len(input_channels) != 2:
            raise ValueError(f'iAFFConcat expects two inputs, got {input_channels}')
        self.input_channels = tuple(int(value) for value in input_channels)
        self.out_channels = sum(self.input_channels)
        self.proj_x = nn.Conv2d(self.input_channels[0], self.out_channels, 1, bias=False)
        self.proj_y = nn.Conv2d(self.input_channels[1], self.out_channels, 1, bias=False)
        self.fusion = iAFF(self.out_channels)

    def forward(self, inputs):
        if not isinstance(inputs, (list, tuple)) or len(inputs) != 2:
            raise ValueError('iAFFConcat expects the two-source list produced by a YOLO fusion layer.')
        x, y = inputs
        if x.shape[-2:] != y.shape[-2:]:
            raise ValueError(f'Spatial mismatch before iAFF: {x.shape} vs {y.shape}')
        x = self.proj_x(x)
        y = self.proj_y(y)
        return self.fusion(x, y)

In [ ]:
# Focused unit checks for the new fusion behavior before model integration.
unit_fusion = iAFF(24)
unit_x = torch.randn(2, 24, 10, 10, requires_grad=True)
unit_y = torch.randn(2, 24, 10, 10, requires_grad=True)
unit_output = unit_fusion(unit_x, unit_y)
assert unit_output.shape == unit_x.shape
unit_output.mean().backward()
assert any(parameter.grad is not None for parameter in unit_fusion.parameters())

unit_adapter = iAFFConcat([16, 24])
adapter_output = unit_adapter([torch.randn(2, 16, 10, 10), torch.randn(2, 24, 10, 10)])
assert adapter_output.shape == (2, 40, 10, 10)
print('iAFF unit checks: forward shape, projection shape and backward gradient OK')

In [ ]:
# Clone YOLO26s, replace only the probed P3/P4 Concat modules, then transfer all compatible tensors.
# This is notebook-local and does not modify the installed Ultralytics package.
custom_model = YOLO(MODEL_NAME)
source_state = base_model.model.state_dict()

for spec in fusion_specs:
    old_module = custom_model.model.model[spec['index']]
    replacement = iAFFConcat(spec['input_channels'])
    # Ultralytics' graph runner and model.info use these layer attributes.
    replacement.i = getattr(old_module, 'i', spec['index'])
    replacement.f = getattr(old_module, 'f', spec['from'])
    replacement.type = getattr(old_module, 'type', replacement.__class__.__name__)
    replacement.np = sum(parameter.numel() for parameter in replacement.parameters())
    custom_model.model.model[spec['index']] = replacement

target_state = custom_model.model.state_dict()
compatible_state = {
    key: value
    for key, value in source_state.items()
    if key in target_state and tuple(value.shape) == tuple(target_state[key].shape)
}
load_report = custom_model.model.load_state_dict(compatible_state, strict=False)
model = custom_model

transferred_tensors = len(compatible_state)
transferred_parameters = sum(value.numel() for value in compatible_state.values())
print(f'Compatible pretrained tensors transferred: {transferred_tensors:,}')
print(f'Compatible pretrained parameter values transferred: {transferred_parameters:,}')
print(f'Newly initialized iAFF/projection tensors: {len(load_report.missing_keys):,}')
print('Unexpected keys:', load_report.unexpected_keys)
print('--- Custom YOLO26s+iAFF architecture ---')
print(model.model)

## 4. Preflight: model.info, iAFF count, GFLOPs, dummy forward và computational graph

In [ ]:
def extract_gflops(info):
    if isinstance(info, (tuple, list)):
        for value in reversed(info):
            try:
                return float(value)
            except (TypeError, ValueError):
                continue
    if isinstance(info, dict):
        for key in ('GFLOPs', 'gflops', 'flops'):
            if key in info:
                try:
                    return float(info[key])
                except (TypeError, ValueError):
                    pass
    return None

iaff_count = sum(
    1 for module in model.model.modules()
    if module.__class__.__name__ == 'iAFF'
)
print('Số module iAFF:', iaff_count)
assert iaff_count > 0, 'iAFF chưa thực sự được chèn vào model'

print('--- model.info() custom model ---')
model.info()
model_info = model.info(verbose=True, imgsz=640)
custom_gflops = extract_gflops(model_info)
print(f'GFLOPs: {custom_gflops:.4f}' if custom_gflops is not None else 'GFLOPs: API không trả về giá trị')

def sum_tensors(value):
    if torch.is_tensor(value):
        return value.float().sum()
    if isinstance(value, (list, tuple)):
        terms = [sum_tensors(item) for item in value]
        return sum(terms) if terms else torch.zeros((), device=next(model.model.parameters()).device)
    if isinstance(value, dict):
        terms = [sum_tensors(item) for item in value.values()]
        return sum(terms) if terms else torch.zeros((), device=next(model.model.parameters()).device)
    return torch.zeros((), device=next(model.model.parameters()).device)

device = next(model.model.parameters()).device
model.model.eval()
with torch.no_grad():
    dummy_output = model.model(torch.zeros(1, 3, 640, 640, device=device))
print('Dummy forward 1x3x640x640: OK')

# Backward probe: ít nhất một parameter của chính iAFF phải nhận gradient.
model.model.train()
model.model.zero_grad(set_to_none=True)
graph_input = torch.randn(2, 3, 640, 640, device=device, requires_grad=True)
graph_output = model.model(graph_input)
graph_loss = sum_tensors(graph_output)
assert graph_loss.requires_grad, 'Dummy output không nằm trong computational graph.'
graph_loss.backward()
iaff_gradients = [
    parameter.grad
    for module in model.model.modules()
    if module.__class__.__name__ == 'iAFF'
    for parameter in module.parameters()
    if parameter.grad is not None
]
assert iaff_gradients and all(torch.isfinite(gradient).all() for gradient in iaff_gradients)
print(f'Computational graph check: OK ({len(iaff_gradients)} iAFF parameter gradients)')
model.model.eval()

## 5. Train E2 — chỉ validation, không chạy test

In [ ]:
train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    optimizer='auto',
    seed=0,
    deterministic=True,
    patience=15,
    project=str(OUTPUT_DIR),
    name='helmet_yolo26s_iaff_dataset2',
    save=True,
    save_period=10,
    plots=True,
    exist_ok=False,
)

In [ ]:
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy best.pt của E2: {BEST_MODEL_PATH}')
best_model = YOLO(BEST_MODEL_PATH)
assert getattr(best_model, 'task', None) == 'detect'
iaff_count_best = sum(
    1 for module in best_model.model.modules()
    if module.__class__.__name__ == 'iAFF'
)
assert iaff_count_best > 0, 'best.pt không chứa iAFF.'
print(f'Đã load E2 checkpoint: {BEST_MODEL_PATH}')
print(f'Số module iAFF trong best.pt: {iaff_count_best}')

## 6. Validation metrics trên split `val`

In [ ]:
def safe_f1(precision_value, recall_value):
    denominator = precision_value + recall_value
    return 2.0 * precision_value * recall_value / denominator if denominator else 0.0

def extract_metrics(split_metrics, split_name):
    summary = {
        'Precision': float(split_metrics.box.mp),
        'Recall': float(split_metrics.box.mr),
        'mAP50': float(split_metrics.box.map50),
        'mAP50-95': float(split_metrics.box.map),
    }
    summary['F1'] = safe_f1(summary['Precision'], summary['Recall'])
    summary['by_class'] = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision_value, recall_value, ap50_value, ap5095_value = split_metrics.box.class_result(class_id)
        class_summary = {
            'Precision': float(precision_value),
            'Recall': float(recall_value),
            'AP50': float(ap50_value),
            'AP50-95': float(ap5095_value),
        }
        class_summary['F1'] = safe_f1(class_summary['Precision'], class_summary['Recall'])
        summary['by_class'][class_name] = class_summary

    print(f'--- {split_name} overall ---')
    for metric_name in ('Precision', 'Recall', 'F1', 'mAP50', 'mAP50-95'):
        print(f'{metric_name:10s}: {summary[metric_name]:.4f}')
    print('--- Metrics theo class ---')
    for class_name, values in summary['by_class'].items():
        print(f'{class_name:24s} | ' + ' | '.join(f'{key}={values[key]:.4f}' for key in ('Precision', 'Recall', 'F1', 'AP50', 'AP50-95')))
    return summary

metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split='val',
    imgsz=640,
    device=0,
    plots=True,
)
validation_summary = extract_metrics(metrics, 'E2 validation')
no_helmet_metrics = validation_summary['by_class']['rider_without_helmet']
print('--- rider_without_helmet ---')
for metric_name in ('Precision', 'Recall', 'F1', 'AP50', 'AP50-95'):
    print(f'{metric_name:10s}: {no_helmet_metrics[metric_name]:.4f}')

# E2 không chạy test set.
test_metrics = None
test_summary = None

## 7. Results, confusion matrix và các curve bắt buộc

In [ ]:
PLOT_SPECS = [
    ('results.png', 'Training results'),
    ('confusion_matrix.png', 'Confusion matrix'),
    ('confusion_matrix_normalized.png', 'Normalized confusion matrix'),
    ('BoxPR_curve.png', 'Box PR curve'),
    ('BoxF1_curve.png', 'Box F1 curve'),
    ('BoxP_curve.png', 'Box Precision curve'),
    ('BoxR_curve.png', 'Box Recall curve'),
]

def find_experiment_plot(filename):
    candidates = list(EXPERIMENT_DIR.rglob(filename)) if EXPERIMENT_DIR.exists() else []
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

for filename, title in PLOT_SPECS:
    plot_path = find_experiment_plot(filename)
    if plot_path is None:
        print(f'Không tìm thấy {filename} trong {EXPERIMENT_DIR}.')
        continue
    plt.figure(figsize=(16, 10) if filename == 'results.png' else (10, 8))
    plt.imshow(Image.open(plot_path))
    plt.axis('off')
    plt.title(f'{title} — {plot_path.parent.name}')
    plt.show()

In [ ]:
def get_confusion_matrix_array(split_metrics):
    confusion_object = getattr(split_metrics, 'confusion_matrix', None)
    matrix = getattr(confusion_object, 'matrix', None)
    if matrix is None:
        return None
    matrix = np.asarray(matrix)
    expected_shape = (len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1)
    return matrix if matrix.shape == expected_shape else None

confusion_matrix_array = get_confusion_matrix_array(metrics)
if confusion_matrix_array is None:
    print('Không đọc được confusion matrix object trực tiếp; không tự tạo số liệu lỗi.')
else:
    confusion_labels = CLASS_NAMES + ['background']
    confusion_table = pd.DataFrame(
        confusion_matrix_array,
        index=[f'predicted:{label}' for label in confusion_labels],
        columns=[f'actual:{label}' for label in confusion_labels],
    )
    display(confusion_table)
    label_to_index = {label: index for index, label in enumerate(confusion_labels)}

    def error_count(actual_label, predicted_label):
        return float(confusion_matrix_array[label_to_index[predicted_label], label_to_index[actual_label]])

    error_rows = [
        {'Error': 'rider_without_helmet -> rider_with_helmet', 'Count': error_count('rider_without_helmet', 'rider_with_helmet')},
        {'Error': 'rider_without_helmet -> background', 'Count': error_count('rider_without_helmet', 'background')},
        {'Error': 'background -> rider_without_helmet', 'Count': error_count('background', 'rider_without_helmet')},
    ]
    display(pd.DataFrame(error_rows))

## 8. Complexity và inference speed trên cùng validation images

In [ ]:
valid_img_dir = ROOT_DIR / 'DataSet2' / 'DataSet2' / 'valid' / 'images'
valid_images = sorted(
    [str(path) for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG') for path in valid_img_dir.glob(pattern)]
)
if not valid_images:
    raise FileNotFoundError(f'Không tìm thấy ảnh validation: {valid_img_dir}')

def model_gflops(yolo_model):
    # Ultralytics returns the info tuple when verbose=True in the installed API.
    return extract_gflops(yolo_model.info(verbose=True, imgsz=640))

def benchmark_inference(yolo_model, image_paths):
    for image_path in image_paths[:2]:
        yolo_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
    elapsed = []
    for image_path in image_paths:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        yolo_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed.append(time.perf_counter() - start_time)
    milliseconds = float(np.mean(elapsed) * 1000.0)
    return milliseconds, 1000.0 / milliseconds if milliseconds else 0.0

speed_images = valid_images[:min(50, len(valid_images))]
if not BASELINE_BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy baseline best.pt: {BASELINE_BEST_MODEL_PATH}')
baseline_model_for_complexity = YOLO(BASELINE_BEST_MODEL_PATH)
iaff_ms, iaff_fps = benchmark_inference(best_model, speed_images)
baseline_ms, baseline_fps = benchmark_inference(baseline_model_for_complexity, speed_images)

complexity = pd.DataFrame([
    {
        'Model': 'YOLO26s',
        'Parameters': sum(parameter.numel() for parameter in baseline_model_for_complexity.model.parameters()),
        'GFLOPs': model_gflops(baseline_model_for_complexity),
        'Model size (MB)': BASELINE_BEST_MODEL_PATH.stat().st_size / (1024 ** 2),
        'Inference (ms/image)': baseline_ms,
        'FPS': baseline_fps,
    },
    {
        'Model': 'YOLO26s+iAFF',
        'Parameters': sum(parameter.numel() for parameter in best_model.model.parameters()),
        'GFLOPs': model_gflops(best_model),
        'Model size (MB)': BEST_MODEL_PATH.stat().st_size / (1024 ** 2),
        'Inference (ms/image)': iaff_ms,
        'FPS': iaff_fps,
    },
])
display(complexity)

## 9. So sánh trực tiếp baseline trên cùng validation set

In [ ]:
if not BASELINE_BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy baseline best.pt: {BASELINE_BEST_MODEL_PATH}')

baseline_comparison_metrics = baseline_model_for_complexity.val(
    data=str(DATA_YAML_PATH),
    split='val',
    imgsz=640,
    device=0,
    plots=False,
)
baseline_summary = extract_metrics(baseline_comparison_metrics, 'YOLO26s baseline comparison')
iaff_summary = validation_summary

def comparison_values(summary):
    no_helmet = summary['by_class']['rider_without_helmet']
    return {
        'Precision': summary['Precision'],
        'Recall': summary['Recall'],
        'F1': summary['F1'],
        'mAP50': summary['mAP50'],
        'mAP50-95': summary['mAP50-95'],
        'NoHelmet Precision': no_helmet['Precision'],
        'NoHelmet Recall': no_helmet['Recall'],
        'NoHelmet F1': no_helmet['F1'],
        'NoHelmet AP50': no_helmet['AP50'],
        'NoHelmet AP50-95': no_helmet['AP50-95'],
    }

baseline_values = comparison_values(baseline_summary)
iaff_values = comparison_values(iaff_summary)
comparison = pd.DataFrame({
    'Metric': list(baseline_values.keys()),
    'YOLO26s': list(baseline_values.values()),
    'YOLO26s+iAFF': [iaff_values[key] for key in baseline_values],
})
comparison['Delta'] = comparison['YOLO26s+iAFF'] - comparison['YOLO26s']
display(comparison)

## 10. Quyết định E2 theo kết quả thực tế

Không hard-code kết quả. Quy tắc dưới đây chỉ đọc các số liệu vừa tính: ưu tiên ba tiêu chí `mAP50-95`, `NoHelmet AP50`, `NoHelmet Recall`, đồng thời yêu cầu phần lớn metric chính tăng.

In [ ]:
priority_metrics = ['mAP50-95', 'NoHelmet AP50', 'NoHelmet Recall']
main_metrics = list(baseline_values.keys())
priority_improvements = {metric: iaff_values[metric] > baseline_values[metric] for metric in priority_metrics}
main_improvement_count = sum(iaff_values[metric] > baseline_values[metric] for metric in main_metrics)
priority_improvement_count = sum(priority_improvements.values())

if priority_improvement_count >= 2 and main_improvement_count >= 6:
    e2_decision = 'KEEP iAFF'
else:
    e2_decision = 'REJECT iAFF'

print(f'Priority improvements: {priority_improvements}')
print(f'Main metrics improved: {main_improvement_count}/{len(main_metrics)}')
print(f'E2 decision: {e2_decision}')

## 11. Final audit trước khi lưu notebook

In [ ]:
assert MODEL_NAME == 'yolo26s.pt'
assert CLASS_NAMES == EXPECTED_CLASS_NAMES
assert TRAIN_CONFIG == {
    'epochs': 50, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'patience': 15,
}
assert iaff_count_best > 0
assert test_metrics is None and test_summary is None
assert EXPERIMENT_NAME == 'helmet_yolo26s_iaff_dataset2'
assert BASELINE_EXPERIMENT_NAME not in str(BEST_MODEL_PATH)
assert BEST_MODEL_PATH == OUTPUT_DIR / 'helmet_yolo26s_iaff_dataset2' / 'weights' / 'best.pt'

forbidden_modules = {'P2', 'DySample', 'MultiSEAM', 'SEAM', 'SCConv', 'CBAM', 'ECA', 'AFPN'}
custom_module_names = {module.__class__.__name__ for module in best_model.model.modules()}
assert not (custom_module_names & forbidden_modules), custom_module_names & forbidden_modules
assert all(spec['stride'] in (8, 16) for spec in fusion_specs)
assert set(comparison['Metric']) == set(baseline_values)

print('Final audit passed.')
print(f'iAFF modules       : {iaff_count_best}')
print(f'Fusion points      : {[spec["index"] for spec in fusion_specs]} (runtime-discovered P3/P4)')
print(f'Validation only    : split={"val"!r}; test disabled')
print(f'E2 checkpoint      : {BEST_MODEL_PATH}')
print(f'Baseline untouched : {BASELINE_BEST_MODEL_PATH}')